## Cart Pole Balancing

The CartPole-v1 environment simulates a balancing act of a pole, hinged at its bottom to a cart, which moves left and right along a track. Balancing the pole upright is carried out by applying to the cart one unit of force—to the right or to the left—at a time. The pole, acting as a pendulum in this environment, starts upright within a small random angle.

The  goal is to keep the pendulum from falling over to either side for as long as possible, that is, up to 500 time steps. For every time step that the pole remains upright, we get a reward of +1, so the maximum total reward is 500. The episode will end prematurely if one of the following occurs during the run: 

- The angle of the pole from the vertical position exceeds 15 degrees.
- The cart's distance from the center exceeds 2.4 units.

Consequently, the total reward in these cases will be smaller than 500.

The expected action value in this simulation is an integer of one of the two following
values:
- 0: Push the cart to the left
- 1: Push the cart to the right

The observation object contains four floats that describe the following information:
- Cart position (between -2.4 and 2.4)
- Cart velocity (between -Inf and Inf)
- Pole angle (between -41.8° and 41.8°)
- Pole velocity at the pole's tip (between -Inf and Inf)

For example, we could have [ 0.33676587, 0.3786464, -0.00170739, -0.36586074].

see https://github.com/openai/gym/wiki/CartPole-v0#solved-requirements


In [1]:
pip install gymnasium stable-baselines3 #takes long time because it installs torch

In [3]:
import gymnasium as gym
import time

In [ ]:
env = gym.make('CartPole-v1', render_mode="human") 
env.reset()
for _ in range(500):#number of steps per episode
    env.render() #display on external window
    action=env.action_space.sample() #make random action
    obs, rew, done, _,info=env.step(action) # take a random action
    time.sleep(0.02)
    if done:
       env.reset()
env.close()

### Evolutionary Approach

We want to EA to generate solutions to the CartPole problem. In the earlier problem MountainCar we generated actions randomly. It would be better to generate actions based on the response the environment. This can be obtained from the observation object returned from the gym step. 

We need to use the observation returned and generate actions based on it. We need to map the observations to actions. One way of doing this mapping is through a neural network. The learning task here can be thought of as teaching a controller to balance the pole by mapping the four available inputs—cart position, cart velocity, pole angle, and pole velocity—into the appropriate action at each time step.

A neural network, such as a Multilayer Perceptron (MLP), can implement complex mappings between its inputs and outputs. This mapping is done with the aid of the network parameters, namely, the weights
and biases of the active nodes in the network, as well as the transfer functions that are implemented by these nodes.

We will use a network with a single hidden layer of two nodes. In addition, the input layer consists of four nodes, one for each of the input values provided by the environment, while the output layer has a single node since we only have one output value for the action to be taken (0 or 1).

![MLP](mlp.png)

The weights in the neural network (MLP) must be trained so that the accurate association between the inputs (observations) and output (actions) can be learned. The common method of training is to use the `backpropagation` algorithm by presenting the associated input and correct output pairs. However in the problem the correct output is not given, so the `backpropagation` method can't be used.

This means that instead of using the conventional training algorithms, we need a method that will allow us to find the weights and biases based on the results that are obtained by running the environment's episodes. 

This is exactly the kind of optimization that genetic algorithms are good at: finding a set of parameters that will give us the best results, as long as you have a way to evaluate and compare these results. 

### Multilayer Perceptron

We begin by defining the MLP (neural networks) structure.

In [4]:
from sklearn.neural_network import MLPRegressor

INPUTS = 4
HIDDEN_LAYER = 2
OUTPUTS = 1
NUM_OF_PARAMS = INPUTS * HIDDEN_LAYER + HIDDEN_LAYER * OUTPUTS + \
                HIDDEN_LAYER + OUTPUTS

def initMlp(netParams):
        """
        initializes a MultiLayer Perceptron (MLP) Regressor with the desired network architecture (layers)
        and network parameters (weights and biases).
        :param netParams: a list of floats representing the network parameters (weights and biases) of the MLP
        :return: initialized MLP Regressor
        """

        # create the initial MLP:
        mlp = MLPRegressor(hidden_layer_sizes=(HIDDEN_LAYER,), max_iter=1)

        # This will initialize input and output layers, and nodes weights and biases:
        # we are not otherwise interested in training the MLP here, hence the settings max_iter=1 above
        mlp.fit(np.random.uniform(low=-1, high=1, size=INPUTS).reshape(1, -1), np.ones(OUTPUTS))

        # weights are represented as a list of 2 ndarrays:
        # - hidden layer weights: INPUTS x HIDDEN_LAYER
        # - output layer weights: HIDDEN_LAYER x OUTPUTS
        numWeights = INPUTS * HIDDEN_LAYER + HIDDEN_LAYER * OUTPUTS
        weights = np.array(netParams[:numWeights])
        mlp.coefs_ = [
            weights[0:INPUTS * HIDDEN_LAYER].reshape((INPUTS, HIDDEN_LAYER)),
            weights[INPUTS * HIDDEN_LAYER:].reshape((HIDDEN_LAYER, OUTPUTS))
        ]

        # biases are represented as a list of 2 ndarrays:
        # - hidden layer biases: HIDDEN_LAYER x 1
        # - output layer biases: OUTPUTS x 1
        biases = np.array(netParams[numWeights:])
        mlp.intercepts_ = [biases[:HIDDEN_LAYER], biases[HIDDEN_LAYER:]]

        return mlp

### Individuals in EA

The Individual in the EA are the weights (and biases) of the MLP (neural network). In order to use neural network of the Multilayer Perceptron type, the set of parameters that we will need to optimize are the network's weights and biases, as follows:

- Input layer: This layer does not participate in the network mapping; instead, it receives the input values and forwards them to every neuron in the next layer. Therefore, no parameters are needed for this network.
- Hidden layer: Each node in this layer is fully connected to each of the inputs, and therefore requires four weights in addition to a single bias value.
- Output layer: The single node in this layer is connected to each of the nodes in the hidden layer, and therefore requires four weights in addition to a single bias value.

In total, we have 4x2+2=10 weight values and 3 bias values (for each hidden node and output node) for the MLP. Therefore, each potential solution can be represented as a list of 15 float values. This is the values for the EA individual.

## Fitness

We use the MLP as the controller for the cart pole during one episode. The resulting total reward of the episode is used as the score value for this solution. This problem requires to maximize the score that's achieved.

We define a getScore function to get the fitness scores.

In [5]:
import numpy as np

def getScore(render, netParams):
    """
    calculates the score of a given solution, represented by the list of float-valued network parameters,
    by creating a corresponding MLP Regressor, initiating an episode of the Cart-Pole environment and
    running it with the MLP controlling the actions, while using the observations as inputs.
    Higher score is better.
    :param netParams: a list of floats representing the network parameters (weights and biases) of the MLP
    :return: the calculated score value
    """

    mlp = initMlp(netParams)

    env.reset()

    actionCounter = 0
    totalReward = 0
    observation,_ = env.reset()
    action = int(mlp.predict(observation.reshape(1, -1)) > 0)

    while True:
        actionCounter += 1
        if render:
            env.render()
            time.sleep(0.02)
        observation, reward, done, _,info = env.step(action)
        totalReward += reward

        if done:
            break
        else:
            action = int(mlp.predict(observation.reshape(1, -1)) > 0)
            #print(action)
    env.reset()
    return totalReward

## Evolutionary Solution



In [6]:
from deap import base
from deap import creator
from deap import tools

import random
import numpy as np

In [7]:
# Genetic Algorithm constants:
POPULATION_SIZE = 20
P_CROSSOVER = 0.9  # probability for crossover
P_MUTATION = 0.5   # probability for mutating an individual
MAX_GENERATIONS = 10
HALL_OF_FAME_SIZE = 1
CROWDING_FACTOR = 10.0
# set the random seed:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
# weight and bias values are bound between -1 and 1:
BOUNDS_LOW, BOUNDS_HIGH = -1.0, 1.0  # boundaries for all dimensions

### Fitness and Individuals

In [8]:
toolbox = base.Toolbox()

# define a single objective, maximizing fitness strategy:
creator.create("FitnessMax", base.Fitness, weights=(1.0,))

# create the Individual class based on list:
creator.create("Individual", list, fitness=creator.FitnessMax)

# helper function for creating random real numbers uniformly distributed within a given range [low, up]
# it assumes that the range is the same for every dimension
def randomFloat(low, up):
## YOURCODE

# create an operator that randomly returns a float in the desired range:
## YOURCODE

# create an operator that fills up an Individual instance:
## YOURCODE

# create an operator that generates a list of individuals:
## YOURCODE

# fitness calculation using the CrtPole class:
def getCartScore(individual):
    return getScore(False, individual),


toolbox.register("evaluate", getCartScore)

### Genetic Operators

In [9]:
# genetic operators:
## YOURCODE select, mate, mutate

### EA Algorithm

In [10]:
from deap import algorithms
import warnings
warnings.filterwarnings('ignore') 

env = gym.make('CartPole-v1', render_mode="human") 

population = toolbox.populationCreator(n=POPULATION_SIZE)

# prepare the statistics object:
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("max", np.max)
stats.register("avg", np.mean)

# define the hall-of-fame object:
hof = tools.HallOfFame(HALL_OF_FAME_SIZE)

# perform the EA  flow with hof feature added:
## YOURCODE

# print best solution found:
best = hof.items[0]
print()
print("Best Solution = ", best)
print("Best Score = ", best.fitness.values[0])
print()
env.close()

gen	nevals	max	avg  
0  	20    	29 	11.65
1  	19    	14 	9.75 
2  	20    	37 	11.95
3  	20    	49 	16.15
4  	15    	58 	14.75
5  	20    	190	38.05
6  	20    	500	84.1 
7  	20    	454	79.7 
8  	20    	241	67.15
9  	20    	384	119.25
10 	20    	500	137.25

Best Solution =  [0.3520098989484017, 0.39254202638501423, -0.9357995121919245, -0.08615487747668132, -0.06215078017383352, -0.9681265873451513, 0.6278526422957378, -0.5584465907998264, -0.370126205119696, -0.8179965858425394, -0.2702027688730946, 0.828002384154264, 0.6713785425064096]
Best Score =  500.0



In [11]:
env = gym.make('CartPole-v1', render_mode="human") 
getScore(True, best)
env.close()

### Challenge Solution Mark
The challenge is considered solved if the reward meets or exceed an average of 490.0 over 100 consecutive trials. We can check whether the requirement is fulfilled by running 100 consecutive tests using our best individual and averaging the results from all the tests.

In [12]:
scores = []
for test in range(100):
    scores.append(getScore(False,best))
print("scores = ", scores)
print("Avg. score = ", sum(scores) / len(scores))

scores =  [144.0, 270.0, 442.0, 274.0, 142.0, 352.0, 113.0, 464.0, 280.0, 214.0, 310.0, 202.0, 216.0, 206.0, 151.0, 206.0, 144.0, 123.0, 202.0, 208.0, 353.0, 209.0, 220.0, 327.0, 299.0, 170.0, 121.0, 125.0, 427.0, 260.0, 198.0, 362.0, 220.0, 314.0, 500.0, 306.0, 240.0, 208.0, 139.0, 125.0, 227.0, 208.0, 370.0, 133.0, 227.0, 171.0, 128.0, 366.0, 228.0, 206.0, 341.0, 168.0, 152.0, 327.0, 132.0, 236.0, 232.0, 194.0, 143.0, 254.0, 189.0, 332.0, 210.0, 365.0, 234.0, 228.0, 395.0, 141.0, 173.0, 294.0, 308.0, 186.0, 192.0, 93.0, 271.0, 216.0, 500.0, 133.0, 174.0, 144.0, 294.0, 230.0, 274.0, 354.0, 170.0, 151.0, 182.0, 260.0, 206.0, 157.0, 119.0, 222.0, 196.0, 141.0, 192.0, 115.0, 155.0, 330.0, 312.0, 214.0]
Avg. score =  232.81
